In [1]:
# Cell 1
import os
import time
import json
import torch
import faiss
import warnings
from tqdm.notebook import tqdm
import ipywidgets as widgets
from IPython.display import display, HTML

# Suppress HuggingFace warnings for a clean demo
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sentence_transformers import SentenceTransformer

import pandas as pd
import csv

# Load resources with proper quoting to avoid "Issue A"
resources_df = pd.read_csv("../data/resources.csv", quoting=csv.QUOTE_ALL)

def show_resources(question):
    # Search for the topic in the CSV (simple keyword search)
    for index, row in resources_df.iterrows():
        if row['topic'].lower() in question.lower():
            display(HTML(f"<h4>🎓 Extra Learning for: {row['topic']}</h4>"))
            
            # Show Image
            display(HTML(f"<img src='{row['image_url']}' width='300' style='border-radius:10px;'>"))
            
            # Show Video
            display(HTML(f"<br><iframe width='560' height='315' src='{row['youtube_url']}' frameborder='0' allowfullscreen></iframe>"))
            return True
    return False

# Setup device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Loading TISD Model on {device.upper()}... Please wait.")

# Paths
VECTORSTORE_DIR = "../vectorstore"
MODELS_DIR = "../models"
INDEX_FILE = os.path.join(VECTORSTORE_DIR, "faiss_index.bin")
METADATA_FILE = os.path.join(VECTORSTORE_DIR, "chunk_metadata.json")
ADAPTER_PATH = os.path.join(MODELS_DIR, "tisd-tinyllama-lora")
BASE_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 1. Load Retriever Memory
index = faiss.read_index(INDEX_FILE)
with open(METADATA_FILE, "r", encoding="utf-8") as f:
    document_chunks = json.load(f)
encoder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# 2. Load Generator Brain
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map={"": device}
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print("✅ TISD System Fully Loaded and Ready!")

Loading TISD Model on MPS... Please wait.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ TISD System Fully Loaded and Ready!


In [2]:
 # Cell 2
def chat_with_tisd(question, top_k=3):
    start_time = time.time()
    
    # 1. Retrieve context
    question_vector = encoder.encode([question], normalize_embeddings=True)
    distances, indices = index.search(question_vector, top_k)
    
    retrieved_contexts = [document_chunks[indices[0][i]]['chunk_text'] for i in range(top_k)]
    combined_context = " ".join(retrieved_contexts)
    
    # 2. TWEAKED PROMPT: Added the "I don't know" safety valve
    system_prompt = (
        "You are TISD, a friendly teacher for grade 1 to 4 students. "
        "Read the Context below. If the Context contains the answer to the Question, explain it simply. "
        "If the Context does NOT contain the answer, you MUST say: 'I'm sorry, I cannot find that in our textbooks yet!'"
    )
    
    # Notice we format this so TinyLlama doesn't get confused
    full_prompt = f"<|system|>\n{system_prompt}</s>\n<|user|>\nContext: {combined_context}\nQuestion: {question}</s>\n<|assistant|>\n"
    
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    
    # 3. Generate Answer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,   # Increased from 150 so it doesn't get cut off
            temperature=0.1,      # Lowered to 0.1 to make it strictly follow our instructions
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.15
        )
    
    # 4. Clean up the output string
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = generated_text.split("<|assistant|>\n")[-1].strip()
    
    # Failsafe cleanups just in case TinyLlama acts stubborn
    if answer.startswith("Answer:"):
        answer = answer.replace("Answer:", "").strip()
        
    elapsed_time = time.time() - start_time
    
    return answer, combined_context, elapsed_time

In [ ]:
# Cell 3
# Create UI Elements
title = widgets.HTML("<h2>🎒 TISD: The Grade 1-4 Teaching Assistant</h2>")
description = widgets.HTML("<i>Ask a question based on your NCERT textbooks!</i>")

text_input = widgets.Text(
    value='',
    placeholder='e.g., What is the sun?',
    description='Student:',
    layout=widgets.Layout(width='80%')
)

button = widgets.Button(
    description='Ask TISD',
    button_style='info',
    tooltip='Click to send your question'
)

output_area = widgets.Output()

def on_button_click(b):
    with output_area:
        output_area.clear_output()
        question = text_input.value
        if not question.strip():
            print("Please type a question!")
            return
            
        print("🤔 Thinking...\n")
        
        # Run the RAG pipeline
        answer, context, latency = chat_with_tisd(question)
        
        output_area.clear_output()
        
        # Display styled output
        display(HTML(f"<b>🧑‍🎓 Student:</b> {question}"))
        display(HTML(f"<div style='background-color:#f0f8ff; padding:10px; border-radius:5px;'><b>🤖 TISD Teacher:</b> {answer}</div>"))
        
        
        display(HTML(f"<p style='color:gray; font-size:0.8em; margin-top:10px;'>"
                     f"⏱️ Answered in {latency:.2f} seconds using M4 MPS backend.</p>"))
        
        show_resources(question)
        
        # FIXED: Building the accordion without the "with" context manager
        acc = widgets.Accordion(children=[widgets.HTML(f"<small>{context}</small>")])
        acc.set_title(0, '📚 View the textbook passage TISD read to answer this')
        display(acc)

button.on_click(on_button_click)

# Display the UI
ui = widgets.VBox([title, description, widgets.HBox([text_input, button]), output_area])
display(ui)